# super_Voz - StyleTTS2 no Colab

Use **Executar tudo**. O notebook monta o Drive, clona/atualiza o repositorio do GitHub em `/content/Super_voz`, entra em `super_Voz`, confere os caminhos do Drive definidos no YAML, copia os audios para a pasta local do Colab e so depois executa o pipeline StyleTTS2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_DIR = Path("/content/Super_voz/super_Voz")
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-u", "scripts/diagnose_colab.py", "--config", "styletts2_colab_config.yml"], check=True)

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/warllemedicao/Super_voz.git"
REPO_DIR = Path("/content/Super_voz")
PROJECT_DIR = REPO_DIR / "super_Voz"

if REPO_DIR.exists():
    print(f"Repositorio ja existe em {REPO_DIR}. Atualizando...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
if not PROJECT_DIR.exists():
    raise RuntimeError(f"Pasta do projeto nao encontrada: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print(f"Diretorio ativo: {PROJECT_DIR}")

## Configuracao

A configuracao principal fica em `styletts2_colab_config.yml`. Os caminhos do Drive sao lidos desse arquivo. Antes de treinar, a proxima celula mostra quais pastas existem e quais serao usadas.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)

In [ ]:
from pathlib import Path
import os
import yaml

PROJECT_DIR = Path("/content/Super_voz/super_Voz")
CONFIG_PATH = PROJECT_DIR / "styletts2_colab_config.yml"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Config nao encontrada: {CONFIG_PATH}\n"
        f"Diretorio atual: {Path.cwd()}\n"
        "Execute a celula de clone/atualizacao antes desta ou confirme se o GitHub tem a pasta super_Voz."
    )
os.chdir(PROJECT_DIR)
print(f"Diretorio ativo: {PROJECT_DIR}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("Drive project dir:", cfg.get("drive_project_dir"))

print("\nAudios brutos candidatos:")
for item in cfg.get("raw_audio_candidates", []):
    path = Path(item)
    count = sum(1 for p in path.rglob("*") if p.is_file()) if path.exists() else 0
    print(f"- {path} | existe={path.exists()} | arquivos={count}")

print("\nAudios processados candidatos:")
for item in cfg.get("processed_audio_candidates", []):
    path = Path(item)
    has_train = (path / "train.txt").exists()
    count = sum(1 for p in path.rglob("*") if p.is_file()) if path.exists() else 0
    print(f"- {path} | existe={path.exists()} | arquivos={count} | train.txt={has_train}")

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_DIR = Path("/content/Super_voz/super_Voz")
CONFIG_PATH = PROJECT_DIR / "styletts2_colab_config.yml"
SCRIPT_PATH = PROJECT_DIR / "scripts" / "run_colab_styletts2.py"
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Pasta do projeto nao encontrada: {PROJECT_DIR}")
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config nao encontrada: {CONFIG_PATH}")
if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Script nao encontrado: {SCRIPT_PATH}")
os.chdir(PROJECT_DIR)
print(f"Diretorio ativo: {PROJECT_DIR}")

cmd = [sys.executable, "-u", "scripts/run_colab_styletts2.py", "--config", "styletts2_colab_config.yml"]
print("$ " + " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
code = proc.wait()
if code != 0:
    raise RuntimeError(f"Pipeline super_Voz falhou com codigo {code}. Veja as linhas acima para a causa real.")